# Scenario analysis — Drought event

This notebook is the **data provenance** for one `drought` scenario in the CRMA
scenario simulator. It reproduces, from the raw **ARCO** (Analysis-Ready
Cloud-Optimized) stores, the evidence-card values the trainee sees — proving each
number came from real reanalysis/forecast data subjected to the operational
transform, not a hand-typed card.

**The journey** (every section below is one hop):

```
ARCO store  ->  subset / reanalysis transform  ->  admin-1 zonal reduce
            ->  Gaussian soft-bin  ->  evidence-node probability vector
            ->  Julia / RxInfer BN posterior  ->  CRMA state
```

Method references: `bn-ibf/drought_ibf/README.md` and (flood) the 11+1-event
run record `bn-ibf/flood_ibf/2026-06-09-11flood_events_run_crma_record.md`.
The flood analogue lives in the sibling `flood_template.ipynb`.

In [ ]:
# --- parameters (papermill injects per-scenario overrides here) ---
EVENT_ID        = "kenya_asal_drought_2020"
BN_IBF_ROOT     = None          # None -> sibling bn-ibf checkout; or set a path
FORECAST_SOURCE = "auto"        # auto | ifs_ens_wb2 | ecmwf_icechunk (flood)
WINDOW_PRE      = 10            # days before peak (flood window)
WINDOW_POST     = 5            # days after peak  (flood window)
RP_YEARS        = 5
COST_LOSS_RATIO = 0.20
RUN_LIVE        = True          # False -> narrate from cache only, no cloud/Julia

In [ ]:
import os, sys
if BN_IBF_ROOT:
    os.environ["BN_IBF_ROOT"] = BN_IBF_ROOT
sys.path.insert(0, os.path.dirname(os.path.abspath("_scenario_nb_lib.py")) or ".")
import _scenario_nb_lib as L
import pandas as pd
from IPython.display import Markdown, display

scenario = L.load_scenario(EVENT_ID)
display(Markdown(
    f"## {scenario['title']}\n\n"
    f"- **Hazard / country / admin-1**: {scenario['hazard']} · "
    f"{scenario['country']} · {scenario['admin1']}  (`{scenario['gid_1']}`)\n"
    f"- **Forecastability**: {scenario['forecastability']}\n"
    f"- **Peak (hidden until debrief)**: {scenario['peak']['date']}\n"
    f"- **EM-DAT key**: {scenario.get('emdat_event_key','—')}\n\n"
    f"> {scenario['brief_outcome_free']}"
))

## 1. The ARCO datasets — open the cloud-optimized stores

The drought pipeline is the monthly analogue of flood: ERA5 SPI reanalysis (obs),
SEAS5 51-member SPI-3 forecast (6 lead months), and ERA5 fitted SPI return-period
thresholds. All anonymous-readable. Opened lazily — structure only.

In [ ]:
prep = L.import_prep("drought")          # operational drought_data_prep module

obs = prep.open_zarr_anon(prep.SPI_OBS_PREFIX)            # ERA5 SPI reanalysis
print("ERA5 SPI obs:", dict(obs.sizes))
forecast = prep.open_icechunk_anon(prep.SEAS5_SPI3_PREFIX)   # SEAS5 51-member SPI3
print("SEAS5 SPI3 forecast:", dict(forecast.sizes))
rp = prep.open_icechunk_anon(prep.SPI_RP_PREFIX)         # fitted SPI RP thresholds
print("ERA5 SPI RP thresholds:", dict(rp.sizes))

## 2. Reanalysis / transform -> admin-1 evidence

Per monthly init the prep computes: `current_spi3` (latest ERA5 month ≤ target),
`deficit_prob = P(SPI ≤ -1.0)` across SEAS5 leads, `tail = p5(ens_min SPI)`,
spatial coverage, and the 6-month SPI trend — reduced to 227 admin-1 polygons.
Threshold direction is **SPI ≤ RP** (deficit), the mirror of flood's TP ≥ RP.

In [ ]:
inits = L.drought_inits(scenario)          # monthly cursors from the JSON rounds
print("Init months (rounds):", inits)

csvs = {}
for m in inits:
    if RUN_LIVE:
        csvs[m] = L.run_drought_prep(scenario, m, rp_years=RP_YEARS)
    else:
        csvs[m] = L.event_cache(EVENT_ID) / "bn_inputs" / f"drought_inputs_{m}.csv"
print(csvs)

In [ ]:
first = next(iter(csvs.values()))
row = L.boundary_row(first, scenario["gid_1"])
cols = [c for c in ["name", "country", "current_spi3", "current_spi3_category",
                    "forecast_deficit_prob", "ens_min_spi", "spatial_coverage",
                    "spi3_trend", "trend_slope_spi_per_month", "target_season",
                    "target_date"] if c in row.index]
display(row[cols].to_frame("value"))

## 3. Scalar -> evidence node (Gaussian soft-binning)

Same soft-evidence mechanism as flood, drought node set
(`cur/def/spa/trn/tail`). The probability vector is the virtual evidence fed to
each BN parent.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
pairs = [   # (short soft_bin code, the prep CSV column it bins)
    ("cur",  row.get("current_spi3", np.nan)),
    ("def",  row.get("forecast_deficit_prob", np.nan)),
    ("tail", row.get("ens_min_spi", np.nan)),
    ("spa",  row.get("spatial_coverage", np.nan)),
    ("trn",  row.get("trend_slope_spi_per_month", np.nan)),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 6)); axes = axes.ravel()
computed = {}
for ax, (node, val) in zip(axes, pairs):
    if pd.isna(val):
        ax.set_visible(False); continue
    labels = L.DROUGHT_NODE_LABELS[node]
    vec = L.plot_soft_bin(prep, node, float(val), labels, ax=ax)
    computed[node] = labels[int(vec.argmax())]
for ax in axes[len(pairs):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()
print("argmax states:", computed)

### 3b. The CDI node — *virtual* evidence, not a BN parent

Drought scenarios add a **Combined Drought Indicator** card
(`evidence_type: virtual`, `bn_node: cdi_class`). The CDI is **not** a parent of
`risk_level`; it multiplies the risk posterior (a 14×5 noisy-likelihood) =
Pearl's virtual evidence. This is the key conceptual difference from the flood
parents — it sharpens the posterior without sitting in the DAG.

In [ ]:
cdi_card = next((c for c in scenario["evidence_cards"]
                 if c.get("bn_node") == "cdi_class"), None)
if cdi_card:
    display(Markdown(
        f"**{cdi_card['label']}**  — type *{cdi_card['evidence_type']}*\n\n"
        f"- value_by_date: {cdi_card.get('value_by_date', {})}\n"
        f"- teaching note: {cdi_card.get('teaching_note','')}\n\n"
        "Applied multiplicatively to P(risk_level) after the parent inference, "
        "then renormalized — it cannot create a state the parents gave zero mass."))
else:
    print("This scenario has no CDI card.")

## 4. Julia / RxInfer drought BN inference

The CSV goes to `drought_bn_ibf_v1.jl` (RxInfer; **Julia output is
authoritative**, the Python pgmpy path is sanity-only per the drought README).
5 parents -> `risk_level` -> cost-loss CRMA.

In [ ]:
outs = {}
for m, csv in csvs.items():
    outs[m] = L.run_drought_bn(scenario, csv) if RUN_LIVE else (
        L.event_cache(EVENT_ID) / "output" / f"drought_bn_v1_{m}.csv")
first_out = next(iter(outs.values()))
brow = L.boundary_row(first_out, scenario["gid_1"])
show = [c for c in ["risk_level", "crma_state", "traffic_light",
                    "risk_minimal", "risk_low", "risk_moderate", "risk_high",
                    "risk_extreme", "crma_explanation"] if c in brow.index]
display(brow[show].to_frame("posterior / CRMA"))

## 5. Round-by-round CRMA timeline (lead -> onset -> peak)

Slow-onset droughts give long lead. We show the CRMA call at each round's init
month so the value of the ~6-month seasonal signal is explicit.

In [ ]:
rows = []
for m, out in outs.items():
    try:
        r = L.boundary_row(out, scenario["gid_1"])
        rows.append({"init": m, "risk_level": r.get("risk_level"),
                     "crma_state": r.get("crma_state")})
    except KeyError:
        pass
display(pd.DataFrame(rows))
for rd in scenario["rounds"]:
    print(f"  R{rd['round']} {rd['cursor_date']}  reveal={rd.get('reveal_evidence')}"
          f"  | engine: {rd.get('engine_state','')}")

## 6. Counterfactual & 7. Debrief

In [ ]:
cf = scenario.get("counterfactual", {})
db = scenario.get("debrief", {})
display(Markdown(
    f"**Counterfactual** (virtual-evidence node "
    f"`{cf.get('virtual_evidence_node','—')}`): {cf.get('prompt','')}\n\n"
    f"> {cf.get('narrative','')}\n\n---\n\n"
    f"**Debrief** — loss `{db.get('loss_markdown','—')}` · "
    f"RK month {db.get('rk_month','—')} · EM-DAT {L.emdat_key(scenario)}\n\n"
    "**Reconstruction quiz**:\n" +
    "\n".join(f"  - {q}" for q in db.get("reconstruction_quiz", []))
))